# Classify 500 customer reviews via the ai_query() SQL function, compare cost vs a per-row Python loop, and configure a token cap

The product team's email subject line is "Quick favor: can you sentiment-tag 500 reviews by EOD?" Yes. You have one session. Two implementations: the Python loop you reach for first, and the ai_query SQL version you should have reached for first. Side by side. Same input. Different cost. Different latency. One paragraph that picks the winner and the reasoning that fits behind "EOD on 500 rows."

The hands-on work:

- Load a 500-row synthetic customer-reviews fixture into a Delta table at `workspace.default.labrun_batch_inference.reviews`.
- Implement a per-row Python loop that calls the Foundation Model API LLM endpoint per row, captures sentiment plus extracted topic, and persists to `predictions_python` (on the first 50 rows to keep the loop tractable inside a session).
- Implement the same workflow as a single SQL `CREATE TABLE AS SELECT` with `ai_query('databricks-meta-llama-3-3-70b-instruct', ..., responseFormat => 'STRUCT<sentiment STRING, topic STRING>', maxTokens => 50)` against all 500 rows.
- Capture wall-clock time and total tokens billed for both approaches into a single `approach_comparison` table.
- Configure the `maxTokens` cap and prove it caps the bill via a deliberate runaway-generation demo (a 2000-word-essay prompt with maxTokens unset vs capped) persisted to `runaway_demo`.

Estimated time: 60 minutes. Cost: $0.03 to $0.10 per session (1000 Foundation Model API calls total across two batches plus the runaway demo).


In [ ]:
# NBVAL_SKIP
# Pinned dependencies. ai_query() runs server-side; the only client-side need
# is the Databricks SDK to drive SQL and the per-row Python loop.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 mlflow==2.20.0 langchain==0.3.7 databricks-langchain==0.1.1


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers (schema, table, volume) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream).

import atexit
import getpass
import json
import re
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "ai-query-batch-inference-and-cost-control"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_batch_inference"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

REVIEWS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.reviews"
PREDICTIONS_PYTHON_FQN = f"{LAB_SCHEMA_FQN}.predictions_python"
PREDICTIONS_AIQUERY_FQN = f"{LAB_SCHEMA_FQN}.predictions_aiquery"
APPROACH_COMPARISON_FQN = f"{LAB_SCHEMA_FQN}.approach_comparison"
RUNAWAY_DEMO_FQN = f"{LAB_SCHEMA_FQN}.runaway_demo"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
PYTHON_LOOP_SAMPLE = 50  # Python loop only runs over the first 50 rows; SQL ai_query runs all 500
TOTAL_REVIEWS = 500


In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, and
# resolve the Starter SQL warehouse. This cell must succeed before the
# manifest cell runs anything.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit SQL to the warehouse and return {columns, rows}."""
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s"
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. Per
# RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution if any
# tagged resources from a prior session exist.

USER_EMAIL = CALLER_USER

CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=RUNAWAY_DEMO_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {RUNAWAY_DEMO_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=APPROACH_COMPARISON_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {APPROACH_COMPARISON_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=PREDICTIONS_AIQUERY_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {PREDICTIONS_AIQUERY_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=PREDICTIONS_PYTHON_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {PREDICTIONS_PYTHON_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=REVIEWS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {REVIEWS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter for UC + MLflow + serving + vector search resources."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_view":
            run_sql(f"DROP VIEW IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().delete_registered_model(name=rid)
            except Exception:
                pass
        elif rtype == "mlflow_experiment":
            try:
                import mlflow
                exp = mlflow.get_experiment_by_name(rid)
                if exp is not None:
                    mlflow.delete_experiment(exp.experiment_id)
            except Exception:
                pass
        elif rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.delete(name=rid)
                deadline = time.time() + 600
                while time.time() < deadline:
                    try:
                        w.serving_endpoints.get(name=rid)
                    except Exception:
                        return
                    time.sleep(5)
            except Exception:
                pass
        elif rtype == "vector_search_index":
            try:
                w.vector_search_indexes.delete_index(index_name=rid)
            except Exception:
                pass
        elif rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.delete_endpoint(endpoint_name=rid)
            except Exception:
                pass
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype in ("uc_managed_table", "uc_view"):
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, table = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.tables "
                    f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                    f"AND table_name = '{table}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume":
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, volume = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.volumes "
                    f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                    f"AND volume_name = '{volume}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            parts = rid.split(".")
            if len(parts) != 2:
                return False
            catalog, schema = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.schemata "
                    f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().get_registered_model(name=rid)
                return True
            except Exception:
                return False
        if rtype == "mlflow_experiment":
            try:
                import mlflow
                return mlflow.get_experiment_by_name(rid) is not None
            except Exception:
                return False
        if rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.get(name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_index":
            try:
                w.vector_search_indexes.get_index(index_name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.get_endpoint(endpoint_name=rid)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Stand up the schema and the 500-row reviews fixture

The fixture is generated inline so the lab is self-contained: 500 short synthetic customer-review strings spanning three categories (delivery, quality, support) and three sentiments (positive, neutral, negative). Each review gets a numeric `review_id` and a `review_text` column. The CTAS writes the rows directly into the Delta table at `workspace.default.labrun_batch_inference.reviews`.

The approach-comparison table is also created up front (empty) so both batches can `INSERT INTO` it later without ordering surprises.


In [ ]:
# NBVAL_SKIP
# Build the 500-row reviews fixture inline, then write it to a Delta table.

import random

random.seed(7)

CATEGORIES = [
    ("delivery", [
        "Package arrived two days early and undamaged.",
        "Delivery was on time but the box was crushed.",
        "Driver left the box in the rain.",
        "Shipping update tracker stopped working halfway.",
        "Showed up on the right day in good shape.",
    ]),
    ("quality", [
        "Product is exactly as described and feels solid.",
        "Stitching came apart after one wash.",
        "Color is a little duller than the photos.",
        "Build quality is fine for the price.",
        "Stopped working after three days of use.",
    ]),
    ("support", [
        "Support answered within an hour and fixed it.",
        "Got bounced between three reps before anyone helped.",
        "Reply took five days; finally got a refund.",
        "Agent was polite but could not solve the problem.",
        "Chat agent resolved it in five minutes.",
    ]),
]

reviews = []
for rid in range(1, TOTAL_REVIEWS + 1):
    cat, samples = random.choice(CATEGORIES)
    text = random.choice(samples)
    reviews.append((rid, text))

print(f"Generated {len(reviews)} synthetic reviews across delivery/quality/support.")

# Stand up the lab schema and tag it for the orphan scan.
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# Create the reviews table.
run_sql(
    f"CREATE OR REPLACE TABLE {REVIEWS_TABLE_FQN} ("
    "review_id BIGINT, review_text STRING)"
)
run_sql(f"ALTER TABLE {REVIEWS_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# Build the INSERT in chunks so the SQL statement does not exceed the warehouse
# statement-length ceiling. 100 rows per INSERT comfortably fits.
CHUNK = 100
for i in range(0, len(reviews), CHUNK):
    batch = reviews[i:i + CHUNK]
    values = ", ".join(
        f"({rid}, '{txt.replace(chr(39), chr(39) * 2)}')"
        for rid, txt in batch
    )
    run_sql(f"INSERT INTO {REVIEWS_TABLE_FQN} VALUES {values}")

row_count = int(run_sql(f"SELECT count(*) FROM {REVIEWS_TABLE_FQN}")["rows"][0][0])
print(f"Reviews table populated: {row_count} rows in {REVIEWS_TABLE_FQN}")

# Approach-comparison table sits empty until each batch fills its own row.
run_sql(
    f"CREATE OR REPLACE TABLE {APPROACH_COMPARISON_FQN} ("
    "approach STRING, wall_clock_s DOUBLE, tokens_billed BIGINT, "
    "cost_usd DOUBLE, rows_processed BIGINT)"
)
run_sql(f"ALTER TABLE {APPROACH_COMPARISON_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
print(f"Approach-comparison table ready: {APPROACH_COMPARISON_FQN}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: reviews table has exactly 500 unique rows with non-null review_text.


def checkpoint_1(session):
    try:
        total = int(run_sql(
            f"SELECT count(*) FROM {REVIEWS_TABLE_FQN}"
        )["rows"][0][0])
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=(
                f"Could not read {REVIEWS_TABLE_FQN}: {exc!r}. "
                f"Did the setup cell run end-to-end?"
            ),
        )
    if total != TOTAL_REVIEWS:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{REVIEWS_TABLE_FQN} has {total} rows; expected exactly {TOTAL_REVIEWS}."
            ),
        )
    nulls = int(run_sql(
        f"SELECT count(*) FROM {REVIEWS_TABLE_FQN} "
        f"WHERE review_text IS NULL OR review_text = ''"
    )["rows"][0][0])
    if nulls > 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{nulls} rows have null or empty review_text; expected zero."
            ),
        )
    distinct = int(run_sql(
        f"SELECT count(DISTINCT review_id) FROM {REVIEWS_TABLE_FQN}"
    )["rows"][0][0])
    if distinct != TOTAL_REVIEWS:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"review_id is not unique across the table; "
                f"{distinct} distinct ids found out of {total} rows."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The setup cell creates the schema, the reviews table, and inserts 500 rows in chunks of 100. If the checkpoint fails on row count, the most common cause is an interrupted INSERT loop. Re-run the setup cell; `CREATE OR REPLACE TABLE` plus the chunked INSERT is idempotent.

</details>

<details><summary>Hint 2 (direction)</summary>

Confirm the table is actually populated before running the checkpoint. Open a SQL cell and run `SELECT count(*) FROM workspace.default.labrun_batch_inference.reviews`. If the count is anything other than 500, the setup cell was interrupted. Re-run it end-to-end.

If `review_id` is not unique, the INSERT loop ran twice over the same range; `CREATE OR REPLACE` at the top of the setup cell resets the table.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The setup cell already contains the full solution. To confirm everything landed:

```python
print(int(run_sql(f"SELECT count(*) FROM {REVIEWS_TABLE_FQN}")["rows"][0][0]))
print(int(run_sql(f"SELECT count(DISTINCT review_id) FROM {REVIEWS_TABLE_FQN}")["rows"][0][0]))
print(int(run_sql(f"SELECT count(*) FROM {REVIEWS_TABLE_FQN} WHERE review_text IS NULL OR review_text = ''")["rows"][0][0]))
```

All three should print 500, 500, 0.

</details>


**Wallet check.** Zero LLM calls so far. The setup cell ran five `INSERT INTO ... VALUES (...)` statements against the Starter SQL warehouse and one `CREATE SCHEMA` plus two `CREATE OR REPLACE TABLE` calls. Warehouse time is a few seconds; tokens billed: 0. The cost meter starts moving in Task 2.


## Task 2: Run the Python-loop baseline and capture wall-clock + tokens billed

The Python loop is the approach most engineers reach for first: one `w.serving_endpoints.query()` call per review, parse the JSON response, append the sentiment plus topic, persist. To keep the lab tractable inside a single session you run the loop over the FIRST 50 rows (the `PYTHON_LOOP_SAMPLE` constant). The ai_query SQL approach in Task 3 will run over all 500 because the SQL engine parallelizes.

Each LLM call costs about 80 input tokens plus 20 output tokens. The pole here is wall-clock time: every call is a synchronous HTTP round-trip plus tokenization plus generation, so 50 sequential calls add up to one to two minutes on a 70B endpoint.

The result lands in `predictions_python` with columns `review_id`, `sentiment`, `topic`, and a row in `approach_comparison` with `approach='python_loop'`.


In [ ]:
# NBVAL_SKIP
# Task 2: per-row Python loop calling the Foundation Model API LLM endpoint.

from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

PYTHON_LOOP_SYSTEM = (
    "You classify a customer review. Respond with strict JSON on a single line: "
    "{\"sentiment\": \"positive|neutral|negative\", \"topic\": \"<one or two words>\"}. "
    "No prose. No code fences."
)

run_sql(
    f"CREATE OR REPLACE TABLE {PREDICTIONS_PYTHON_FQN} ("
    "review_id BIGINT, sentiment STRING, topic STRING)"
)
run_sql(f"ALTER TABLE {PREDICTIONS_PYTHON_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# Fetch the first PYTHON_LOOP_SAMPLE review rows in id order.
rows_for_python = run_sql(
    f"SELECT review_id, review_text FROM {REVIEWS_TABLE_FQN} "
    f"ORDER BY review_id LIMIT {PYTHON_LOOP_SAMPLE}"
)["rows"]
print(f"Pulled {len(rows_for_python)} rows for the Python loop.")


def classify_review(review_text):
    resp = w.serving_endpoints.query(
        name=LLM_ENDPOINT,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=PYTHON_LOOP_SYSTEM),
            ChatMessage(role=ChatMessageRole.USER, content=review_text),
        ],
        max_tokens=40,
        temperature=0.0,
    )
    raw = (resp.choices[0].message.content or "").strip()
    usage = getattr(resp, "usage", None)
    tokens = 0
    if usage is not None:
        tokens = int(getattr(usage, "total_tokens", 0) or 0)
    return raw, tokens


def parse_prediction(raw):
    """Best-effort parse of the JSON response into (sentiment, topic)."""
    sentiment = None
    topic = None
    try:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            obj = json.loads(match.group(0))
            sentiment = str(obj.get("sentiment", "")).strip().lower()
            topic = str(obj.get("topic", "")).strip().lower()
    except Exception:
        pass
    if sentiment not in ("positive", "neutral", "negative"):
        sentiment = "neutral"
    if not topic:
        topic = "unknown"
    return sentiment, topic


# YOUR CODE: record start = time.time() before the loop
# YOUR CODE: total_tokens = 0; predictions = []
# YOUR CODE: for review_id, review_text in rows_for_python:
# YOUR CODE:     raw, tokens = classify_review(review_text)
# YOUR CODE:     sentiment, topic = parse_prediction(raw)
# YOUR CODE:     predictions.append((review_id, sentiment, topic))
# YOUR CODE:     total_tokens += tokens
# YOUR CODE: wall_clock = time.time() - start
# YOUR CODE: build a single INSERT INTO predictions_python VALUES (...) for all rows
# YOUR CODE: build the approach_comparison INSERT with
# YOUR CODE:   approach='python_loop', wall_clock_s=wall_clock, tokens_billed=total_tokens,
# YOUR CODE:   cost_usd=total_tokens * 0.0000003 (pay-per-token estimate),
# YOUR CODE:   rows_processed=len(predictions)
# YOUR CODE: print a one-line wallet check showing tokens, wall-clock, and cents


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: predictions_python has PYTHON_LOOP_SAMPLE rows with sentiment
# in the allowed set and a non-null topic; approach_comparison has a row for
# approach='python_loop' with non-zero wall_clock and tokens_billed.


def checkpoint_2(session):
    try:
        row = run_sql(
            f"SELECT count(*), "
            f"sum(CASE WHEN sentiment IN ('positive','neutral','negative') THEN 1 ELSE 0 END), "
            f"sum(CASE WHEN topic IS NULL OR topic = '' THEN 1 ELSE 0 END) "
            f"FROM {PREDICTIONS_PYTHON_FQN}"
        )["rows"][0]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=(
                f"Could not read {PREDICTIONS_PYTHON_FQN}: {exc!r}. "
                f"Did the Python loop's INSERT run?"
            ),
        )
    total = int(row[0])
    valid_sent = int(row[1] or 0)
    null_topic = int(row[2] or 0)
    if total != PYTHON_LOOP_SAMPLE:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{PREDICTIONS_PYTHON_FQN} has {total} rows; "
                f"expected exactly {PYTHON_LOOP_SAMPLE} (the Python loop sample size)."
            ),
        )
    if valid_sent != total:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{total - valid_sent} rows have a sentiment outside "
                f"{{positive, neutral, negative}}. Re-check parse_prediction."
            ),
        )
    if null_topic > 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{null_topic} rows have a null or empty topic. "
                f"The fallback in parse_prediction should set 'unknown' on parse failure."
            ),
        )
    try:
        comp = run_sql(
            f"SELECT wall_clock_s, tokens_billed, rows_processed "
            f"FROM {APPROACH_COMPARISON_FQN} WHERE approach = 'python_loop'"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not read approach_comparison: {exc!r}",
        )
    if not comp:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison has no row for approach='python_loop'. "
                f"Add the comparison INSERT at the end of Task 2."
            ),
        )
    wall_clock_s, tokens_billed, rows_processed = comp[0]
    if not wall_clock_s or float(wall_clock_s) <= 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison.wall_clock_s for python_loop is {wall_clock_s!r}; "
                f"expected a positive number captured via time.time() deltas."
            ),
        )
    if not tokens_billed or int(tokens_billed) <= 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison.tokens_billed for python_loop is {tokens_billed!r}; "
                f"capture resp.usage.total_tokens per call and sum."
            ),
        )
    if int(rows_processed or 0) != PYTHON_LOOP_SAMPLE:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison.rows_processed for python_loop is "
                f"{rows_processed!r}; expected {PYTHON_LOOP_SAMPLE}."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The loop is a straightforward `for review_id, review_text in rows_for_python`. The two captures that matter are `start = time.time()` before the loop and `total_tokens += tokens` inside it. Build a list of prediction tuples, then a single chunked INSERT at the end.

</details>

<details><summary>Hint 2 (direction)</summary>

```
start = time.time()
total_tokens = 0
predictions = []
for review_id, review_text in rows_for_python:
    raw, tokens = classify_review(review_text)
    sentiment, topic = parse_prediction(raw)
    predictions.append((review_id, sentiment, topic))
    total_tokens += tokens
wall_clock = time.time() - start
```

For the INSERT, build a single `VALUES (...), (...), ...` payload from `predictions` (50 rows fits comfortably in one statement). The approach_comparison INSERT is one row: `('python_loop', wall_clock, total_tokens, total_tokens * 0.0000003, len(predictions))`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
start = time.time()
total_tokens = 0
predictions = []
for review_id, review_text in rows_for_python:
    raw, tokens = classify_review(review_text)
    sentiment, topic = parse_prediction(raw)
    predictions.append((review_id, sentiment, topic))
    total_tokens += tokens
wall_clock = time.time() - start

values_sql = ", ".join(
    f"({rid}, '{s}', '{t.replace(chr(39), chr(39) * 2)}')"
    for rid, s, t in predictions
)
run_sql(f"INSERT INTO {PREDICTIONS_PYTHON_FQN} VALUES {values_sql}")

cost_usd = total_tokens * 0.0000003
run_sql(
    f"INSERT INTO {APPROACH_COMPARISON_FQN} VALUES "
    f"('python_loop', {wall_clock}, {total_tokens}, {cost_usd}, {len(predictions)})"
)

print(
    f"python_loop: {len(predictions)} rows, "
    f"{total_tokens} tokens, "
    f"{wall_clock:.1f}s wall-clock, "
    f"about {cost_usd * 100:.2f} cents."
)
```

</details>


**Wallet check.** Fifty FM API calls at roughly 100 tokens each. About 5000 tokens total. Pay-per-token math at the published Llama-3.3-70B rate puts the bill at about 0.15 cents. The pole is wall-clock: every call is sequential because the loop is a single Python thread. The SQL ai_query approach in Task 3 lifts that ceiling.


## Task 3: Run the ai_query() CTAS over all 500 rows with a maxTokens cap

One SQL statement. One CTAS. All 500 rows in parallel on the warehouse. The function signature:

```sql
ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  concat('Classify this review: ', review_text),
  responseFormat => 'STRUCT<sentiment STRING, topic STRING>',
  maxTokens => 50
) AS prediction
```

The `responseFormat` parameter is what coerces the LLM into a structured output. Without it the prediction column is a freeform STRING and parsing downstream is harder. The `maxTokens => 50` caps the bill per row at 50 output tokens, which is plenty for a one-word sentiment plus a one or two word topic.

After the CTAS lands you compute wall-clock from a `time.time()` delta around the `run_sql` call and pull the row count from DESCRIBE HISTORY operationMetrics to estimate tokens billed (50 max-tokens per row times 500 rows is the worst case; actual is usually lower).


In [ ]:
# NBVAL_SKIP
# Task 3: ai_query() CTAS over all 500 rows with the maxTokens cap.

AIQUERY_CTAS = (
    f"CREATE OR REPLACE TABLE {PREDICTIONS_AIQUERY_FQN} AS "
    "SELECT "
    "  review_id, "
    "  ai_query( "
    "    'databricks-meta-llama-3-3-70b-instruct', "
    "    concat('Classify this review with one sentiment (positive, neutral, negative) and a one-word topic. Review: ', review_text), "
    "    responseFormat => 'STRUCT<sentiment STRING, topic STRING>', "
    "    maxTokens => 50 "
    "  ) AS prediction "
    f"FROM {REVIEWS_TABLE_FQN}"
)

# YOUR CODE: start = time.time(); run_sql(AIQUERY_CTAS, wait_seconds=600); wall_clock = time.time() - start
# YOUR CODE: run_sql to tag the new predictions_aiquery table with LAB_TAG_KEY/LAB_TAG_VALUE
# YOUR CODE: pull the produced row count: rows_processed = SELECT count(*) FROM predictions_aiquery
# YOUR CODE: estimate tokens_billed: use 100 tokens per row as the wallet-side estimate
# YOUR CODE:   (the warehouse does not surface per-call token counts here; the brief flags this).
# YOUR CODE: cost_usd = tokens_billed * 0.0000003
# YOUR CODE: INSERT INTO approach_comparison VALUES ('ai_query', wall_clock, tokens_billed, cost_usd, rows_processed)
# YOUR CODE: print the wallet check line and a side-by-side of python_loop vs ai_query.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: predictions_aiquery has exactly 500 rows; approach_comparison
# has a row for approach='ai_query' with non-zero wall_clock and tokens_billed.


def checkpoint_3(session):
    try:
        total = int(run_sql(
            f"SELECT count(*) FROM {PREDICTIONS_AIQUERY_FQN}"
        )["rows"][0][0])
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=(
                f"Could not read {PREDICTIONS_AIQUERY_FQN}: {exc!r}. "
                f"Did the ai_query CTAS run?"
            ),
        )
    if total != TOTAL_REVIEWS:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{PREDICTIONS_AIQUERY_FQN} has {total} rows; "
                f"expected {TOTAL_REVIEWS} (one prediction per review)."
            ),
        )
    try:
        comp = run_sql(
            f"SELECT wall_clock_s, tokens_billed, rows_processed "
            f"FROM {APPROACH_COMPARISON_FQN} WHERE approach = 'ai_query'"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not read approach_comparison: {exc!r}",
        )
    if not comp:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison has no row for approach='ai_query'. "
                f"Add the comparison INSERT at the end of Task 3."
            ),
        )
    wall_clock_s, tokens_billed, rows_processed = comp[0]
    if not wall_clock_s or float(wall_clock_s) <= 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison.wall_clock_s for ai_query is {wall_clock_s!r}; "
                f"capture time.time() around the run_sql(AIQUERY_CTAS) call."
            ),
        )
    if not tokens_billed or int(tokens_billed) <= 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison.tokens_billed for ai_query is {tokens_billed!r}; "
                f"use the 100-tokens-per-row wallet-side estimate the brief documents."
            ),
        )
    if int(rows_processed or 0) != TOTAL_REVIEWS:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison.rows_processed for ai_query is "
                f"{rows_processed!r}; expected {TOTAL_REVIEWS}."
            ),
        )
    if len(run_sql(f"SELECT approach FROM {APPROACH_COMPARISON_FQN}")["rows"]) < 2:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"approach_comparison must have at least two rows by this point "
                f"(python_loop and ai_query). Re-run Task 2 if it was skipped."
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

One `run_sql(AIQUERY_CTAS)` call inside a `time.time()` delta is the whole task. The pre-built `AIQUERY_CTAS` string already contains the right ai_query() signature. Bump the `wait_seconds` argument to 600 because the warehouse can spend 30 to 90 seconds on a cold start before the CTAS begins.

</details>

<details><summary>Hint 2 (direction)</summary>

```
start = time.time()
run_sql(AIQUERY_CTAS, wait_seconds=600)
wall_clock = time.time() - start

run_sql(f"ALTER TABLE {PREDICTIONS_AIQUERY_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

rows_processed = int(run_sql(f"SELECT count(*) FROM {PREDICTIONS_AIQUERY_FQN}")["rows"][0][0])
tokens_billed = rows_processed * 100   # wallet-side estimate per the brief
cost_usd = tokens_billed * 0.0000003
```

Then a single INSERT for the comparison row.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
start = time.time()
run_sql(AIQUERY_CTAS, wait_seconds=600)
wall_clock = time.time() - start

run_sql(
    f"ALTER TABLE {PREDICTIONS_AIQUERY_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

rows_processed = int(run_sql(
    f"SELECT count(*) FROM {PREDICTIONS_AIQUERY_FQN}"
)["rows"][0][0])
tokens_billed = rows_processed * 100
cost_usd = tokens_billed * 0.0000003

run_sql(
    f"INSERT INTO {APPROACH_COMPARISON_FQN} VALUES "
    f"('ai_query', {wall_clock}, {tokens_billed}, {cost_usd}, {rows_processed})"
)

print(
    f"ai_query: {rows_processed} rows, "
    f"{tokens_billed} tokens (estimate), "
    f"{wall_clock:.1f}s wall-clock, "
    f"about {cost_usd * 100:.2f} cents."
)

# Side-by-side.
sxs = run_sql(
    f"SELECT approach, rows_processed, wall_clock_s, tokens_billed, cost_usd "
    f"FROM {APPROACH_COMPARISON_FQN} ORDER BY approach"
)["rows"]
for row in sxs:
    print(row)
```

</details>


**Wallet check.** 500 FM API calls fired in parallel by the warehouse. Roughly 50,000 tokens total at the same per-token rate. About 1.5 cents on the bill. The shift relative to Task 2: about 10x the rows for about 10x the tokens, but the wall-clock typically holds within 2 to 3x because the warehouse parallelizes the calls. That is the canonical batch-inference shape the exam tests.


## Task 4: Prove the maxTokens cap caps the bill via a deliberate runaway-generation demo

Two ai_query() probe calls. Same prompt, same model, different `maxTokens` setting:

- `maxtokens_cap_set=TRUE`: `maxTokens => 30` against `"write a 2000-word essay about Delta Lake"`. The model stops at 30 output tokens regardless of how long the response wants to be.
- `maxtokens_cap_set=FALSE`: no `maxTokens` argument at all against the same prompt. The model generates until it decides to stop, which is typically 500 to 2000 tokens on a 70B for a "2000-word essay" stem.

Persist both rows to `runaway_demo` with columns `(approach STRING, maxtokens_cap_set BOOLEAN, tokens_billed BIGINT, response STRING)`. The checkpoint validates monotonicity: the capped row's `tokens_billed` must be strictly less than the uncapped row's, by at least the 500-token-difference margin the brief documents.

Tokens billed in this task come from `length(response) / 4` as the wallet-side estimate (roughly 4 characters per token for English). The exact billing number is not surfaced by the warehouse here; the brief documents the estimate.


In [ ]:
# NBVAL_SKIP
# Task 4: runaway-generation demo (capped vs uncapped) to prove the maxTokens
# cap caps the bill.

RUNAWAY_PROMPT = "write a 2000-word essay about Delta Lake"

run_sql(
    f"CREATE OR REPLACE TABLE {RUNAWAY_DEMO_FQN} ("
    "approach STRING, maxtokens_cap_set BOOLEAN, tokens_billed BIGINT, response STRING)"
)
run_sql(f"ALTER TABLE {RUNAWAY_DEMO_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# Two probe-call shapes. Both write a single row with the same column layout.
CAPPED_INSERT = (
    f"INSERT INTO {RUNAWAY_DEMO_FQN} "
    "SELECT 'capped' AS approach, "
    "       TRUE AS maxtokens_cap_set, "
    "       CAST(length(response) / 4 AS BIGINT) AS tokens_billed, "
    "       response "
    "FROM ( "
    "  SELECT ai_query( "
    f"    '{LLM_ENDPOINT}', "
    "    'write a 2000-word essay about Delta Lake', "
    "    maxTokens => 30 "
    "  ) AS response "
    ") t"
)

UNCAPPED_INSERT = (
    f"INSERT INTO {RUNAWAY_DEMO_FQN} "
    "SELECT 'uncapped' AS approach, "
    "       FALSE AS maxtokens_cap_set, "
    "       CAST(length(response) / 4 AS BIGINT) AS tokens_billed, "
    "       response "
    "FROM ( "
    "  SELECT ai_query( "
    f"    '{LLM_ENDPOINT}', "
    "    'write a 2000-word essay about Delta Lake', "
    "    maxTokens => 2000 "
    "  ) AS response "
    ") t"
)

# YOUR CODE: run_sql(CAPPED_INSERT, wait_seconds=300)
# YOUR CODE: run_sql(UNCAPPED_INSERT, wait_seconds=600)
# YOUR CODE: SELECT approach, maxtokens_cap_set, tokens_billed, length(response) AS response_chars
# YOUR CODE:   FROM runaway_demo ORDER BY approach
# YOUR CODE: print both rows so the operator sees the capped vs uncapped delta.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: runaway_demo has both rows (capped TRUE and uncapped FALSE),
# and the capped row's tokens_billed is strictly less than the uncapped row's
# by a margin consistent with the 30 vs 2000 maxTokens setting.


def checkpoint_4(session):
    try:
        rows = run_sql(
            f"SELECT approach, maxtokens_cap_set, tokens_billed "
            f"FROM {RUNAWAY_DEMO_FQN} ORDER BY maxtokens_cap_set DESC"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=(
                f"Could not read {RUNAWAY_DEMO_FQN}: {exc!r}. "
                f"Did both INSERTs run?"
            ),
        )
    if len(rows) < 2:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{RUNAWAY_DEMO_FQN} has {len(rows)} row(s); "
                f"expected at least 2 (capped + uncapped)."
            ),
        )
    capped = None
    uncapped = None
    for approach, cap_set, tokens in rows:
        if bool(cap_set) is True:
            capped = int(tokens or 0)
        elif bool(cap_set) is False:
            uncapped = int(tokens or 0)
    if capped is None:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"No row with maxtokens_cap_set=TRUE found in {RUNAWAY_DEMO_FQN}. "
                f"Did the CAPPED_INSERT run?"
            ),
        )
    if uncapped is None:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"No row with maxtokens_cap_set=FALSE found in {RUNAWAY_DEMO_FQN}. "
                f"Did the UNCAPPED_INSERT run?"
            ),
        )
    if capped >= uncapped:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Capped tokens_billed ({capped}) is not strictly less than "
                f"uncapped tokens_billed ({uncapped}). The maxTokens cap should "
                f"hold the capped response well under the uncapped response on "
                f"a 'write a 2000-word essay' prompt. Re-run both INSERTs."
            ),
        )
    if (uncapped - capped) < 50:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Capped vs uncapped margin is only {uncapped - capped} tokens; "
                f"expected at least 50. The uncapped probe may have stopped early; "
                f"re-run UNCAPPED_INSERT."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

Two `run_sql` calls. The pre-built `CAPPED_INSERT` and `UNCAPPED_INSERT` strings already contain the right ai_query() signatures with and without the maxTokens cap. Run the capped one first (it is fast); run the uncapped one second with a higher `wait_seconds` because the 2000-token generation can take a minute.

</details>

<details><summary>Hint 2 (direction)</summary>

```
run_sql(CAPPED_INSERT, wait_seconds=300)
run_sql(UNCAPPED_INSERT, wait_seconds=600)

rows = run_sql(
    f"SELECT approach, maxtokens_cap_set, tokens_billed, length(response) "
    f"FROM {RUNAWAY_DEMO_FQN} ORDER BY approach"
)["rows"]
for r in rows:
    print(r)
```

If the capped row's `tokens_billed` is not less than the uncapped row's, the cap did not bind; re-check that the CAPPED_INSERT contains `maxTokens => 30`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
print("Running capped probe (maxTokens => 30)...")
run_sql(CAPPED_INSERT, wait_seconds=300)
print("Running uncapped probe (maxTokens => 2000)...")
run_sql(UNCAPPED_INSERT, wait_seconds=600)

rows = run_sql(
    f"SELECT approach, maxtokens_cap_set, tokens_billed, length(response) AS response_chars "
    f"FROM {RUNAWAY_DEMO_FQN} ORDER BY approach"
)["rows"]
for approach, cap_set, tokens, chars in rows:
    print(f"  {approach:9s} cap_set={cap_set} tokens_billed={tokens:5d} response_chars={chars}")
```

Expected output: capped tokens around 7 to 10, uncapped tokens 400 to 1500. The bill ratio is the lab's lesson.

</details>


**Wallet check.** Two ai_query() probe calls. The capped one billed about 30 output tokens; the uncapped one billed several hundred to a couple thousand. Session total now lands in the 3 to 10 cent envelope the brief documents. Without the cap on the uncapped call, that final probe alone could have been the most expensive line in the session; that is the lesson.


## Cleanup

Tear down the five lab tables (runaway_demo, approach_comparison, predictions_aiquery, predictions_python, reviews) and the schema (`CASCADE`). The atexit hook fires the same manifest on kernel shutdown, so re-running this cell is idempotent.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. Prints the canonical summary
# from RESOURCE-SAFETY-SPEC Rule 6 and sys.exit(1) on any failure.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about 3 to 10 cents.** Fifty Python-loop LLM calls plus 500 ai_query() calls plus two runaway probes. Roughly 60,000 to 100,000 tokens. Free Edition's daily token quota took a noticeable hit; if you re-run the lab the wallet check at orphan-scan time warns you. Cleanup dropped every lab table and the schema; the metastore is clean again.


## Reflection

These are not graded. They are for you.

1. Walk through the architectural difference between the Python-loop approach and the ai_query CTAS approach. Where does the LLM call happen in each (driver vs warehouse vs serving endpoint)? Why does the SQL approach typically win on wall-clock time for 500+ rows, even though the per-call cost is the same?

2. The maxTokens cap is one cost-control tool. List three other Databricks features that control LLM costs (Section 6 of the exam guide names "Use Databricks features to control LLM costs"). When does each one win?

## Exam-Style Practice

**Question 1 (Domain 4, batch inference workload identification):**

A GenAI engineer needs to classify 100,000 customer reviews into sentiment buckets nightly. The team is choosing between (a) a Python loop in a Lakeflow Job that calls the Foundation Model API per row, (b) a SQL CTAS with `ai_query()` against the reviews Delta table, (c) deploying a Model Serving endpoint and querying it from each row. Which is the right pick?

A. (a) Python loop, because Python gives more error handling per row.
B. (b) ai_query CTAS, because it is the documented Databricks pattern for batch LLM inference over Delta tables.
C. (c) Model Serving endpoint, because every LLM workload should go through Model Serving.
D. All three are equivalent; pick the one the team is most familiar with.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: Python loops drag against Databricks's columnar SQL engine; for 100,000 rows the SQL approach is faster and easier to maintain.
- B is correct: `ai_query()` inside a SQL statement is the canonical batch-inference pattern. It runs on the SQL warehouse, parallelizes across cluster cores, and produces a single CTAS-style output. Section 4 of the exam guide names "Identify batch inference workloads and apply ai_query() appropriately."
- C is wrong: Model Serving endpoints are optimized for low-latency real-time inference, not bulk batch jobs. They scale up and down per request; running a 100,000-row batch through an endpoint is a worse fit than ai_query.
- D is wrong: the three approaches are NOT equivalent on cost, latency, or maintainability for this workload shape.

</details>

**Question 2 (Domain 6, LLM cost control):**

A GenAI engineer's batch sentiment classification job exceeded its daily token budget twice last week. The model occasionally generated 1000+ token responses to short reviews. Which Databricks feature is the right cost control?

A. Switch to a smaller model so each token is cheaper.
B. Set `maxTokens` on the ai_query call to a sentiment-appropriate cap (e.g., 20).
C. Re-train the model to produce shorter responses.
D. Add a regex to truncate the response after the fact.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: switching models is a separate decision; cheaper per token does not solve "the response is 1000 tokens when it should be 20." Smaller model with no maxTokens cap can still overshoot.
- B is correct: `maxTokens` caps the model's output length at the decoder. The model stops generating when the cap is hit; the token bill stops at the cap. This is the canonical cost control for verbose-output failures.
- C is wrong: retraining is not a per-call cost control and is not an option for the Foundation Model API endpoints the engineer does not own.
- D is wrong: truncating the response after the fact does NOT save tokens; the model already generated them and you already paid for them.

</details>
